# 🎯 Workshop Day 3: Evaluation & Optimization

**Agentic RAG Workshop** | Day 3 of 3

---

### 🎯 Today we will learn
1. **Measure RAG Quality** — with RAGAS (4 metrics)
2. **Build an Eval Dataset** — automatically with Gemini
3. **Test the Agent** — Tool Selection, LLM-as-Judge
4. **Improve the Pipeline** — A/B Experiment
5. **Prompt Engineering** — techniques for writing better instructions
6. **Capstone Project** — combine everything from all 3 days ⭐

### 🗺️ 3-Day Overview

```
Day 1 (Data)           Day 2 (Agent)          Day 3 (Evaluation)
─────────────          ─────────────          ─────────────────
Raw → Chunk →          Agent + Tool →         Measure quality (3.1-3.2)
Embed → VectorDB       RAG Agent →            Test the Agent (3.3)
  → Retrieve           Multi-Agent →          Improve (3.4-3.5)
                       Agentic RAG            Capstone ⭐ (3.6)
```

> 💡 Today is about "evaluation" — understanding how good the system is, then making it better

## 📦 Section 0: Install Dependencies

In [ ]:
%%time
pip3 install python-pptx 2>&1 | tail -3 install -q ragas datasets google-adk google-genai \
    sentence-transformers qdrant-client langchain-text-splitters \
    rank_bm25 scikit-learn langchain-google-genai langchain-huggingface

import warnings
warnings.filterwarnings('ignore')
print('✅ Installation complete!')

In [ ]:
!pip install -q langchain-huggingface

In [ ]:
%%time
import os, json, hashlib, random, asyncio
import numpy as np

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ Loaded API Key from Colab Secrets')
except Exception:
    api_key = input('🔑 Please paste your Gemini API Key: ')
    os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

from google import genai
try:
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    resp = client.models.generate_content(model='gemini-2.5-flash', contents='Reply only with OK')
    print(f'✅ API working: {(resp.text.strip() if resp.text else '(no output)')}')
except Exception as e:
    print(f'❌ API Error: {e}')

### Prepare Data + VectorDB (re-use from Day 1-2)

In [ ]:
%%time
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')

documents = {
    'healthcare': 'Siriraj Hospital uses AI to help analyze chest X-ray images, reducing diagnosis time from 30 minutes to 5 minutes. The Deep Learning system detects abnormalities with 95% accuracy, helping doctors screen tuberculosis patients faster and reducing waiting time for results.',
    'banking': 'Kasikornbank developed KADE AI Assistant using RAG to retrieve financial product information and answer customer questions, reducing response time from 5 minutes to 30 seconds. It supports both Thai and English and can automatically handle questions about loans, credit cards, and investments.',
    'education': 'Chulalongkorn University built a RAG system for question-answering from textbooks, converting more than 500 PDF books into vector embeddings. It uses a multilingual model that supports Thai. Students can ask questions and receive answers with references.',
    'agriculture': 'The Department of Agriculture uses AI to analyze drone images to detect plant diseases in rice fields with 92% accuracy, helping reduce crop damage by 40%. It uses Computer Vision and CNN models trained on 50,000 rice disease images.',
    'rag': 'The RAG architecture consists of 3 parts: Retriever searches from the VectorDB, Reader reads documents, and Generator produces the answer. It helps reduce hallucination because the answer comes from real data instead of being made up by the LLM.',
    'embedding': 'Text Embedding converts text into vectors. multilingual-e5-large supports 100 languages including Thai and creates 1024-dimensional vectors. Similar texts will have nearby vectors. Cosine Similarity is used to measure similarity.',
    'kmitl': 'King Mongkut’s Institute of Technology Ladkrabang (KMITL), Faculty of Engineering, developed AI systems for Smart Campus by combining IoT sensors with Machine Learning to analyze building energy usage, reducing electricity costs by 25%. It also developed Thai NLP systems for analyzing student opinions and applied RAG to build an AI Tutor that answers course questions from teaching materials.',
    'nlp': 'Thai NLP is challenging because Thai has no spaces between words, so Word Segmentation tools such as PyThaiNLP are needed. Vowels and tone marks add complexity, so tokenizers must be specially designed.',
    'vectordb': 'Qdrant is a high-performance Vector Database that supports ANN for fast retrieval even with millions of vectors. It supports Cosine, Dot Product, and L2 Distance, and can search together with metadata using payload filters.',
}

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
all_chunks, all_sources = [], []
for src, text in documents.items():
    for chunk in splitter.split_text(text):
        all_chunks.append(chunk)
        all_sources.append(src)

passages = [f'passage: {c}' for c in all_chunks]
embeddings = embed_model.encode(passages, show_progress_bar=True)
VECTOR_SIZE = embeddings.shape[1]

qdrant = QdrantClient(':memory:')
qdrant.create_collection('thai_ai_kb',
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE))
points = [models.PointStruct(id=i, vector=embeddings[i].tolist(),
    payload={'text': all_chunks[i], 'source': all_sources[i]}) for i in range(len(all_chunks))]
qdrant.upsert('thai_ai_kb', points=points)

print(f'✅ Data ready: {len(all_chunks)} chunks, {len(documents)} sources')

# RAG function
def search_qdrant(query, top_k=3):
    q_vec = embed_model.encode(f'query: {query}').tolist()
    results = qdrant.query_points('thai_ai_kb', query=q_vec, limit=top_k).points
    return [{'text': r.payload['text'], 'source': r.payload['source'],
             'score': round(r.score, 4)} for r in results]

def rag_answer(question, top_k=3):
    contexts = search_qdrant(question, top_k)
    context_text = '\n'.join([c['text'] for c in contexts])
    prompt = f'"""Answer the question using only the following information. Respond in Thai. Keep it concise.\n\nInformation:\n{context_text}\n\nQuestion: {question}\nAnswer:"""'
    resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.3))
    return (resp.text.strip() if resp.text else '(no output)'), contexts

# Quick test
ans, ctx = rag_answer('How does AI help Thai doctors?')
print(f'\n🧪 Test RAG: {ans[:80]}...')

---
## 📊 Section 3.1: RAGAS — Measuring RAG Quality

### What is RAGAS?

**RAGAS (Retrieval-Augmented Generation Assessment Suite)** = a framework for measuring RAG quality

```
Input: question + answer + contexts (+ ground_truth)
  ↓ RAGAS computes
Output: score 0.0 - 1.0 for each metric
```

### 4 Main Metrics

| Metric | What does it measure? | Comparison | Need ground_truth? |
|--------|------------------------|------------|:------------------:|
| **Faithfulness** | Is the answer "faithful" to the data? (not hallucinated?) | answer ↔ contexts | ❌ |
| **Answer Relevancy** | Does the answer match the question? | answer ↔ question | ❌ |
| **Context Precision** | Are the retrieved chunks actually relevant? | contexts ↔ ground_truth | ✅ |
| **Context Recall** | Are all important chunks retrieved? | contexts ↔ ground_truth | ✅ |

### 🌡️ Score Interpretation

| Score | Level | What should you do? |
|:-----:|-------|---------------------|
| 0.8-1.0 | 🟢 Excellent | ✅ Ready for use |
| 0.6-0.8 | 🟡 Fair | ⚠️ Can be improved |
| < 0.6 | 🔴 Needs fixing | ❌ Urgently needs improvement |

In [ ]:
%%time
# ─── Build Evaluation Dataset ───
# Must contain: question, answer, contexts, ground_truth

eval_questions = [
    'How does AI help the Thai medical field?',
    'What is the architecture of RAG?',
    'How do Thai banks use AI?',
    'What are the challenges of Thai NLP?',
    'How does Qdrant work?',
]

eval_ground_truths = [
    'Siriraj Hospital uses AI to analyze chest X-rays, reducing diagnosis time from 30 minutes to 5 minutes, with 95% accuracy.',
    'RAG has 3 parts: Retriever searches from VectorDB, Reader reads documents, Generator creates the answer, reducing hallucination.',
    'Kasikornbank developed KADE AI Assistant using RAG to retrieve product information, reducing response time from 5 minutes to 30 seconds.',
    'Thai has no spaces between words, so Word Segmentation such as PyThaiNLP is needed. Vowels and tone marks add complexity.',
    'Qdrant is a high-performance Vector Database that supports ANN for fast search and supports Cosine, Dot Product, and L2.',
]

# Run RAG to collect answers + contexts
eval_answers = []
eval_contexts = []

print('🔄 Generating answers...')
for q in eval_questions:
    ans, ctxs = rag_answer(q)
    eval_answers.append(ans)
    eval_contexts.append([c['text'] for c in ctxs])
    print(f'  ✅ {q[:30]}... → {ans[:40]}...')

print(f'\n📊 Eval Dataset: {len(eval_questions)} questions')

In [ ]:
%%time
# ─── Measure with RAGAS (using Gemini LLM + Local Embeddings) ───
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd

# 1. Use Gemini as the judging LLM (Reasoning)
gemini_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.environ['GOOGLE_API_KEY'],
))

# 2. Use Local Embedding (E5) instead to avoid API 404 issues
local_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-large')
)

# 3. Build Dataset
eval_dataset = Dataset.from_dict({
    'question': eval_questions,
    'answer': eval_answers,
    'contexts': eval_contexts,
    'ground_truth': eval_ground_truths,
})

# 4. Measure with RAGAS
try:
    result = evaluate(
        dataset=eval_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
        llm=gemini_llm,
        embeddings=local_embeddings,
    )

    # Handle results: check both result and result.scores
    final_data = []
    if hasattr(result, 'scores'):
        # If scores is a list, use it directly; if it has to_pandas, call it
        if isinstance(result.scores, list):
            final_data = result.scores
        else:
            final_data = result.scores.to_pandas()
    elif isinstance(result, list):
        final_data = result
    else:
        final_data = [result]

    scores_df = pd.DataFrame(final_data)

    # Compute average of each metric
    avg_scores = scores_df.mean(numeric_only=True)
    for metric, score in avg_scores.items():
        level = '🟢' if score >= 0.8 else ('🟡' if score >= 0.6 else '🔴')
        print(f'  {level} {metric}: {score:.4f}')

except Exception as e:
    print(f'⚠️ RAGAS error: {e}')
    print('💡 If it still errors, you can skip to Section 3.3 (LLM-as-Judge) instead')

### 💡 Observations
- **High Faithfulness** = the Agent is not hallucinating and answers from real data
- **Low Context Precision** = irrelevant chunks were retrieved → try adjusting `chunk_size`, `top_k`
- **Low Context Recall** = important information was not retrieved → try increasing `top_k` or adjusting the embedding

> 🎯 Goal: every metric **≥ 0.7** is considered production-ready

### 🧪 Exercise 3.1
1. Add 3 of your own questions + ground_truth
2. Measure with RAGAS → which metric is the lowest?
3. Analyze why it is low

In [ ]:
# 🧪 Exercise: add questions and measure with RAGAS
# 💡 Hint:
#   1. Add question + ground_truth to the lists
#   2. Loop through rag_answer() to collect answer + contexts
#   3. Build Dataset.from_dict() and then call evaluate()


---
## 🔬 Section 3.2: Automatically Build an Evaluation Dataset

### Why build it automatically?

- Writing ground_truth manually → very slow (5 questions = 30 minutes)
- Use LLM to generate from chunks → get 50 questions in 2 minutes

```
Chunk: "Siriraj uses AI to analyze X-rays with 95% accuracy"
  ↓ Gemini generates
Q: "Which hospital uses AI to analyze X-ray images?"
A: "Siriraj Hospital, with 95% accuracy"
```

In [ ]:
%%time
# ─── Auto-generate Q&A from chunks ───
def generate_qa_from_chunk(chunk_text):
    prompt = f'''From the following information, create 1 question + answer.
Information: {chunk_text}

Reply in JSON: {{"question": "...", "answer": "..."}}
The question must be specific, in Thai, and the answer must reference real information.'''
    try:
        resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
            config=genai.types.GenerateContentConfig(temperature=0.3, response_mime_type='application/json'))
        return json.loads(resp.text)
    except:
        return None

# Build from 8 chunks (select the first chunk of each source)
auto_qa = []
seen_sources = set()
for i, chunk in enumerate(all_chunks):
    src = all_sources[i]
    if src in seen_sources: continue
    seen_sources.add(src)
    qa = generate_qa_from_chunk(chunk)
    if qa:
        qa['source'] = src
        auto_qa.append(qa)
        print(f'  ✅ [{src}] Q: {qa["question"][:50]}...')

print(f'\n📊 Generated {len(auto_qa)} Q&A pairs')

### 💡 Observations
- LLM can generate questions 10x faster than doing it manually
- But you still need to **check quality** — some questions may be too broad
- Use it as a **starting point**, then refine manually

> 🎯 Production: auto-generate 100 questions → manually review → select the best 50

### 🧪 Exercise 3.2
Generate auto Q&A from all chunks, then review the quality

In [ ]:
# 🧪 Exercise
# 💡 Hint: loop through all_chunks and generate Q&A with generate_qa_from_chunk()


---
## 🧪 Section 3.3: Test the Agent

### What do we test?

| Type | What do we test? | Example |
|------|------------------|---------|
| **Tool Selection** | Did it choose the correct tool? | Ask about BMI → call calculate_bmi |
| **Response Quality** | Is the answer good? | Use LLM-as-Judge |
| **Edge Cases** | Does it break? | Weird questions, multiple languages |
| **Guardrails** | Does it refuse inappropriate requests? | Ask about dangerous topics |

In [ ]:
# ─── Build Agent for testing ───
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

def search_knowledge(query: str) -> dict:
    """Search AI-related information in Thailand.
    Args:
        query: The question to search for
    """
    results = search_qdrant(query, top_k=3)
    return {'status': 'success', 'results': results}

def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """Calculate BMI.
    Args:
        weight_kg: Weight in kilograms
        height_cm: Height in centimeters
    """
    h = height_cm / 100
    bmi = weight_kg / (h ** 2)
    return {'bmi': round(bmi, 1)}

test_agent = Agent(
    name='test_agent', model='gemini-2.5-flash',
    instruction='''You are an AI expert. Respond in Thai.
    - Use search_knowledge when asked about AI in Thailand
    - Use calculate_bmi when asked about BMI
    - Answer general questions directly
    - Refuse dangerous or illegal topics''',
    tools=[search_knowledge, calculate_bmi]
)

async def test_agent_call(agent, msg):
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    session_id = f's_{id(agent)}_{id(msg)}'
    await runner.session_service.create_session(
        app_name=agent.name, user_id='tester', session_id=session_id
    )
    content = types.Content(role='user', parts=[types.Part.from_text(text=msg)])
    response, tools = '', []
    async for event in runner.run_async(user_id='tester', session_id=session_id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text: response = p.text
                if p.function_call: tools.append(p.function_call.name)
    return response, tools

print('✅ Test Agent ready')

In [ ]:
# ─── Test Suite: Tool Selection ───
test_cases = [
    {'input': 'Weight 70, height 175, BMI?',      'expected_tool': 'calculate_bmi'},
    {'input': 'How does AI help Thai doctors?',   'expected_tool': 'search_knowledge'},
    {'input': 'Hello',                            'expected_tool': None},
    {'input': 'What is Qdrant?',                  'expected_tool': 'search_knowledge'},
    {'input': 'How is the weather today?',        'expected_tool': None},
]

print('═' * 60)
print('🧪 Test Suite: Tool Selection')
print('═' * 60)

passed = 0
for tc in test_cases:
    resp, tools = await test_agent_call(test_agent, tc['input'])
    actual_tool = tools[0] if tools else None
    ok = actual_tool == tc['expected_tool']
    passed += ok
    status = '✅' if ok else '❌'
    print(f"  {status} '{tc['input'][:25]}...' → expected={tc['expected_tool']}, got={actual_tool}")

print(f'\n📊 Pass rate: {passed}/{len(test_cases)} ({passed/len(test_cases)*100:.0f}%)')

In [ ]:
%%time
# ─── LLM-as-Judge (more strict) ───
def llm_judge(question, answer, max_score=5):
    prompt = f'''Score the answer from 1-{max_score} using these criteria (be strict):
- Correctness: matches the facts, no made-up information (if it invents extra facts, deduct 2 points)
- Referencing: includes sources or specific examples (if none, deduct 1 point)
- Conciseness: answer in bullet points, no more than 5 bullets (if too long, deduct 1 point)
- Relevance: directly answers the question, no going off-topic
- Scope: if information is insufficient, it must say "no information" (if not, deduct 1 point)

Question: {question}
Answer: {answer}

Reply in JSON: {{"score": 1-{max_score}, "reason": "short reason"}}'''
    try:
        resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
            config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
        return json.loads(resp.text)
    except:
        return {'score': 0, 'reason': 'error'}

# Test — use varied questions (easy + hard + edge case)
print('═' * 60)
print('🧪 LLM-as-Judge: Measure answer quality')
print('═' * 60)

judge_questions = [
    'How does AI help Thai doctors?',
    'Compare AI in Thai healthcare vs Thai finance',
    'How many Thai AI companies have already IPOed?',
]

total_score = 0
for q in judge_questions:
    ans, _ = rag_answer(q)
    verdict = llm_judge(q, ans)
    total_score += verdict['score']
    stars = '⭐'*verdict['score'] + '☆'*(5-verdict['score'])
    print(f'  {stars} {verdict["score"]}/5 | {q[:30]}... → {verdict["reason"]}')

avg = total_score / len(judge_questions)
print(f'\n📊 Average: {avg:.1f}/5.0')


### 💡 Observations
- **Tool Selection test** = checks whether the Agent chooses the correct tool (deterministic)
- **LLM-as-Judge** = uses an LLM to evaluate answer quality (no need to write manual rules)
- Use both together → covers both "works correctly" and "answers well"

> ⚠️ LLM-as-Judge is not 100% accurate — use it as a screening tool before human review

### 🧪 Exercise 3.3
Add 5 more test cases, including edge cases (English, weird questions)

In [ ]:
# 🧪 Exercise
# 💡 Hint: add to the test_cases list, then loop through test_agent_call()


---
## ⚡ Section 3.4: Improve the RAG Pipeline

### What can we improve?

| Step | Parameter | Impact |
|------|-----------|--------|
| Chunking | `chunk_size`, `overlap` | Context Precision |
| Retrieval | `top_k`, `alpha` | Context Recall |
| Generation | `temperature`, prompt | Faithfulness, Relevancy |

### A/B Experiment
```
Config A: chunk=100, top_k=3
Config B: chunk=200, top_k=5  ← Better?
Config C: chunk=300, top_k=3
```

In [ ]:
%%time
# ─── A/B Experiment ───
configs = [
    {'name': 'A: small chunks', 'chunk_size': 100, 'overlap': 20, 'top_k': 3},
    {'name': 'B: medium chunks', 'chunk_size': 200, 'overlap': 30, 'top_k': 5},
    {'name': 'C: large chunks', 'chunk_size': 300, 'overlap': 50, 'top_k': 3},
]

test_q = 'How does AI help the Thai medical field?'
test_gt = 'Siriraj Hospital uses AI to analyze X-rays with 95% accuracy'

print('═' * 60)
print('🧪 A/B Experiment: Compare Configs')
print('═' * 60)

for cfg in configs:
    sp = RecursiveCharacterTextSplitter(chunk_size=cfg['chunk_size'], chunk_overlap=cfg['overlap'])
    chunks = []
    for text in documents.values():
        chunks.extend(sp.split_text(text))
    
    # Embed + search
    vecs = embed_model.encode([f'passage: {c}' for c in chunks])
    q_vec = embed_model.encode(f'query: {test_q}')
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity([q_vec], vecs)[0]
    top_idx = np.argsort(sims)[-cfg['top_k']:][::-1]
    top_chunks = [chunks[i] for i in top_idx]
    top_scores = [sims[i] for i in top_idx]
    
    # Check if ground truth info is in retrieved chunks
    gt_found = any('Siriraj' in c or '95%' in c for c in top_chunks)
    
    print(f'\n📋 {cfg["name"]}: {len(chunks)} chunks')
    print(f'   Top scores: {[round(s,3) for s in top_scores]}')
    print(f'   Ground truth found: {"✅" if gt_found else "❌"}')
    print(f'   Best chunk: {top_chunks[0][:60]}...')

### 💡 Observations
- **small chunks** (100) → more chunks, but may split important context
- **large chunks** (300) → fuller context, but may introduce more noise
- **higher top_k** → higher recall, but precision may decrease

> 🎯 There is no single best config for every task — you must **experiment** with real data

### 🧪 Exercise 3.4
Try at least 3 configs → find the best config for this dataset

In [ ]:
# 🧪 Exercise: try different configs
# 💡 Hint: add more configs to the list, then loop through the tests


---
## ✍️ Section 3.5: Prompt Engineering for Agents

### Techniques

| Technique | Example | Reduce Hallucination? |
|-----------|---------|:---------------------:|
| **Role** | "You are an expert..." | ⭐ |
| **Few-shot** | Provide example Q&A | ⭐⭐ |
| **Chain-of-Thought** | "Think step by step..." | ⭐⭐⭐ |
| **Guardrails** | "Do not guess; if you do not know, say so clearly" | ⭐⭐⭐ |
| **Output Format** | "Answer in 3 bullet points" | ⭐ |

In [ ]:
# ─── Before vs After Prompt ───
prompt_before = 'Answer the question'
prompt_after = '''You are an AI expert in Thailand. Respond in Thai.

Rules:
1. Always search for information using search_knowledge before answering
2. Answer only with information found; do not guess or add invented details
3. Cite sources such as [healthcare] [banking]
4. If no information is found, clearly say "No information in the knowledge base"
5. Answer concisely in bullet points, no more than 5 bullets'''

agent_v1 = Agent(name='v1', model='gemini-2.5-flash', instruction=prompt_before, tools=[search_knowledge])
agent_v2 = Agent(name='v2', model='gemini-2.5-flash', instruction=prompt_after, tools=[search_knowledge])

# Use harder questions — the difference in instruction will be clearer
hard_test_questions = [
    'Compare the use of AI in Thai hospitals vs Thai banks',
    'Is Thailand truly a leader in AI in ASEAN?',
    'Fully summarize the challenges of Thai NLP and how to solve them',
]

print('═' * 60)
print('🧪 Before vs After Prompt (hard questions)')
print('═' * 60)

v1_total, v2_total = 0, 0
for q in hard_test_questions:
    r1, _ = await test_agent_call(agent_v1, q)
    r2, _ = await test_agent_call(agent_v2, q)

    j1 = llm_judge(q, r1)
    j2 = llm_judge(q, r2)
    v1_total += j1['score']
    v2_total += j2['score']

    print(f'\n❓ {q}')
    print(f'  V1: {j1["score"]}/5 — {j1["reason"]}')
    print(f'  V2: {j2["score"]}/5 — {j2["reason"]}')

v1_avg = v1_total / len(hard_test_questions)
v2_avg = v2_total / len(hard_test_questions)
print(f'\n{"═"*60}')
print(f'📊 Summary: V1 avg = {v1_avg:.1f}/5  |  V2 avg = {v2_avg:.1f}/5')
diff = v2_avg - v1_avg
if diff > 0:
    print(f'📈 V2 is better than V1 by +{diff:.1f} points — good instructions really help!')
elif diff < 0:
    print(f'📉 V1 is better than V2 — try revising the instructions')
else:
    print(f'➡️ Same score — try adding more difficult questions')


### 💡 Observations
- Good instructions → answers are **referenced, concise, and structured**
- Short instructions → answers may **guess, be too long, and lack references**
- **Guardrails** ("do not guess") help reduce hallucination a lot

> 🎯 Prompt Engineering = the **easiest improvement method with the biggest impact**

### 🧪 Exercise 3.5
Write 3 instruction versions → evaluate with LLM-as-Judge → find the best version

In [ ]:
# 🧪 Exercise
# 💡 Hint: create agent_v3 with a new instruction, then compare scores


---
## 🚀 Section 3.6: Capstone Project ⭐

### Combine everything from all 3 days

```
Day 1: Prepare data → Chunk → Embed → VectorDB
Day 2: Build Agent → Tools → RAG Agent → Multi-Agent
Day 3: Measure with RAGAS → Test → Improve → Measure again
```

### Assignment
1. **Build a RAG Pipeline** from the provided data
2. **Measure quality** with RAGAS + LLM-as-Judge
3. **Improve it** to get better results (chunk_size, prompt, top_k)
4. **Show Before/After results**

In [ ]:
# ─── Capstone: Before → After ───
print('═' * 60)
print('🚀 Capstone: Improve the RAG Pipeline')
print('═' * 60)

# Use mixed questions (easy + hard + edge case) to make the difference visible
capstone_questions = [
    'How does AI help the Thai medical field?',
    'Compare which embedding model is most suitable for Thai',
    'Summarize the pros and cons of RAG compared with fine-tuning',
    'When did Elon Musk visit Thailand?',
    'Explain all AI use cases in Thailand found in the database',
]

# Step 1: Measure baseline (plain rag_answer)
print('\n📊 Step 1: Baseline (plain RAG)')
baseline_scores = []
for q in capstone_questions:
    ans, _ = rag_answer(q)
    verdict = llm_judge(q, ans)
    baseline_scores.append(verdict['score'])
    stars = '⭐'*verdict['score'] + '☆'*(5-verdict['score'])
    print(f'  {stars} {verdict["score"]}/5 | {q[:35]}...')
baseline_avg = sum(baseline_scores) / len(baseline_scores)
print(f'   → Baseline Average: {baseline_avg:.1f}/5.0')

# Step 2: Improve (use Agent + good instruction)
print('\n📊 Step 2: After Optimization (Agent + Good Prompt)')
improved_scores = []
for q in capstone_questions:
    r, _ = await test_agent_call(agent_v2, q)
    verdict = llm_judge(q, r)
    improved_scores.append(verdict['score'])
    stars = '⭐'*verdict['score'] + '☆'*(5-verdict['score'])
    print(f'  {stars} {verdict["score"]}/5 | {q[:35]}...')
improved_avg = sum(improved_scores) / len(improved_scores)
print(f'   → Improved Average: {improved_avg:.1f}/5.0')

# Step 3: Show results
print(f'\n{"═"*60}')
print(f'📊 Summary Before → After')
print(f'{"═"*60}')
print(f'  Baseline:  {"⭐" * round(baseline_avg)} {baseline_avg:.1f}/5.0')
print(f'  Improved:  {"⭐" * round(improved_avg)} {improved_avg:.1f}/5.0')
diff = improved_avg - baseline_avg
if diff > 0:
    print(f'  📈 +{diff:.1f} improved!')
elif diff < 0:
    print(f'  📉 {diff:.1f} got worse — should return to baseline')
else:
    print(f'  ➡️ No change — try adjusting something else')


### 💡 Observations
- **Prompt Engineering** is often the fastest way to see improvement
- **Chunk size** must be tested — there is no fixed formula
- **Measure before and after** is very important — if you do not measure, you do not know whether it actually improved

> 🎯 "What gets measured gets improved" — if you cannot measure it, you cannot improve it

### 🧪 Exercise 3.6
1. Improve at least 2 things (prompt + chunk_size or top_k)
2. Measure Before/After
3. Explain why it improved or did not improve

In [ ]:
# 🧪 Capstone: improve it yourself
# 💡 Hint:
#   1. Change chunk_size → re-embed → re-search → measure again
#   2. Change instruction → create a new agent → measure again
#   3. Change top_k → rag_answer( ,top_k=5) → measure again


---
## 🎯 Summary: What we learned in 3 days

| Day | Content | Main Tools |
|-----|---------|------------|
| **Day 1** | Data Engineering Pipeline | Qdrant, E5-large, BM25 |
| **Day 2** | Agent Building | Google ADK, Tools, Multi-Agent |
| **Day 3** | Evaluation & Optimization | RAGAS, LLM-as-Judge |

### Skills Gained
- ✅ Build a RAG Pipeline from start to finish
- ✅ Build an Agent that can make decisions on its own
- ✅ Measure quality + improve systematically
- ✅ Work in Thai throughout the whole pipeline

### 🛣️ What next?
- **Production**: Deploy on Cloud (GCP, AWS) + database session
- **Advanced**: Fine-tune embeddings for a specific domain
- **Scale**: Qdrant Cloud + horizontal scaling
- **Monitor**: Log + alerts when quality drops

---

**🙏 Thank you for learning together for all 3 days!**

> "The best way to learn AI is to build with AI."